In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import re, pickle
import torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import EsmModel, EsmTokenizer
from sklearn.metrics import roc_auc_score, average_precision_score

BASE_PATH  = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
MODEL_DIR  = "/mnt/volume6/czj/labLGN/LabLZ/models/esm2_650M"
INPUT_CSV  = BASE_PATH + "cell2024_final.csv"
SCORE_CSV  = BASE_PATH + "phase3_esm2_scores.csv"
CACHE_PKL  = BASE_PATH + "esm2_emb_cache.pkl"           # embedding cache
MAX_LEN    = 1022
BATCH_SIZE = 16

ESM2 delta embedding AUROC

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
tokenizer = EsmTokenizer.from_pretrained(MODEL_DIR)
model = EsmModel.from_pretrained(MODEL_DIR).eval().to(device)
if device.type == "cuda":
    model = model.half()
print("Model loaded.")

# Get mutation position from row
def get_pos(row):
    for col in ["Mutation_used", "Mutation"]:
        m = re.match(r'^[A-Z](\d+)[A-Z]$', str(row.get(col, "")))
        if m:
            return int(m.group(1))
    return None

def window(seq, pos):
    # long protein sequences may exceed the model's max length, so we take a window around the mutation position
    if len(seq) <= MAX_LEN:
        return seq
    if pos is None:
        return seq[:MAX_LEN]  # if no mutation position, just take the first MAX_LEN characters
    end   = min(len(seq), pos + MAX_LEN // 2)
    start = max(0, end - MAX_LEN)
    return seq[start:start + MAX_LEN]

Device: cuda


Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: /mnt/volume6/czj/labLGN/LabLZ/esm2_650M
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


In [4]:
df = pd.read_csv(INPUT_CSV)
df_eval = df[df["mutant_sequence"].notna() & df["Mislocalized"].notna()].reset_index(drop=True)
df_eval["_pos"]   = df_eval.apply(get_pos, axis=1)
df_eval["_wtwin"] = df_eval.apply(lambda r: window(r["sequence"],        r["_pos"]), axis=1)
df_eval["_mtwin"] = df_eval.apply(lambda r: window(r["mutant_sequence"], r["_pos"]), axis=1)

n_long = (df_eval["sequence"].str.len() > MAX_LEN).sum()
print(f"Rows to be handled: {len(df_eval)}, from which sequences longer than {MAX_LEN} need a window: {n_long}")

# Get unique sequences to embed, so we don't embed the same sequence multiple times
unique_seqs = sorted(set(df_eval["_wtwin"]) | set(df_eval["_mtwin"]), key=len)

Rows to be handled: 2179, from which sequences longer than 1022 need a window: 73


In [7]:
emb = {}
if os.path.exists(CACHE_PKL):
    with open(CACHE_PKL, "rb") as f:
        emb = pickle.load(f)
    print(f"We already have {len(emb)} embeddings cached.")
todo = [s for s in unique_seqs if s not in emb]
print(f"This time: {len(todo)}")

@torch.inference_mode()
def embed_batch(seqs):
    enc  = tokenizer(seqs, return_tensors="pt", padding=True, add_special_tokens=True)
    ids  = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    out  = model(input_ids=ids, attention_mask=mask).last_hidden_state.float()  # [B,L,H]
    m = mask.clone()
    m[:, 0] = 0                                        # delete CLS
    lengths = mask.sum(dim=1)
    m[torch.arange(m.size(0)), lengths - 1] = 0        # delete EOS
    m = m.unsqueeze(-1).float()
    pooled = (out * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)
    return pooled.cpu().numpy()

for i in tqdm(range(0, len(todo), BATCH_SIZE), desc="ESM2 embedding"):
    batch = todo[i:i + BATCH_SIZE]
    for s, v in zip(batch, embed_batch(batch)):
        emb[s] = v
    if (i // BATCH_SIZE) % 20 == 0:
        with open(CACHE_PKL, "wb") as f:
            pickle.dump(emb, f)
with open(CACHE_PKL, "wb") as f:
    pickle.dump(emb, f)
print(f"embedding finished, altogether {len(emb)} sequences embedded and cached.")

We already have 1952 embeddings cached.
This time: 1074


ESM2 embedding:   0%|          | 0/17 [00:00<?, ?it/s]

ESM2 embedding: 100%|██████████| 17/17 [00:17<00:00,  1.04s/it]

embedding finished, altogether 3026 sequences embedded and cached.


In [8]:
def cosine_distance(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return np.nan if d == 0 else 1.0 - np.dot(a, b) / d

df_eval["esm2_delta_score"] = df_eval.apply(
    lambda r: cosine_distance(emb[r["_wtwin"]], emb[r["_mtwin"]]), axis=1)
df_eval[["Gene", "Variant", "Mislocalized", "esm2_delta_score"]].to_csv(SCORE_CSV, index=False)
print(f"Saved {SCORE_CSV}, valid scores {df_eval['esm2_delta_score'].notna().sum()}/{len(df_eval)}")

Saved /mnt/volume6/czj/labLGN/LabLZ/phase3_esm2_scores.csv, valid scores 2179/2179


In [10]:
print("\n─── baseline assessment with the same n (no addtional_benign) ───")
scored = df.merge(
    df_eval[["Gene", "Variant", "esm2_delta_score"]],
    on=["Gene", "Variant"], how="left")

mask = (scored["mutant_sequence"].notna()
        & scored["Mislocalized"].notna()
        & scored["AlphaMissense score"].notna()
        & scored["esm2_delta_score"].notna())
df_A = scored[mask].copy()
y = df_A["Mislocalized"].astype(int)

def report(name, scores):
    auc, auprc = roc_auc_score(y, scores), average_precision_score(y, scores)
    print(f"{name:15s} AUROC {auc:.4f}  AUPRC {auprc:.4f}")

report("AlphaMissense", df_A["AlphaMissense score"])
report("ESM2 delta",    df_A["esm2_delta_score"])
rng  = np.random.default_rng(0)
aucs = [roc_auc_score(y, rng.random(len(df_A))) for _ in range(10)]
print(f"{'Random':15s} AUROC {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

df_A = scored[mask].copy()
y = df_A["Mislocalized"].astype(int)

print(f"n = {len(df_A)}, positive {int(y.sum())}, negative {int((y == 0).sum())}")


─── baseline assessment with the same n (no addtional_benign) ───
AlphaMissense   AUROC 0.6362  AUPRC 0.1622
ESM2 delta      AUROC 0.5602  AUPRC 0.1475
Random          AUROC 0.5094 ± 0.0229
n = 2053, positive 234, negative 1819
